# Troubleshooting

This tutorial steps through tecnhiques to identify errors and pipeline failures, as well
as avoid common pitfalls setting up executing over multiple processes.


## Things to check if Pydra gets stuck

I There are a number of common gotchas, related to running multi-process code, that can
cause Pydra workflows to get stuck and not execute correctly. If using the concurrent
futures worker (e.g. `worker="cf"`), check these issues first before filing a bug report
or reaching out for help.

### Applying `nest_asyncio` when running within a notebook

When using the concurrent futures worker within a Jupyter notebook you need to apply
`nest_asyncio` with the following lines

In [ ]:
# This is needed to run parallel workflows in Jupyter notebooks
import nest_asyncio

nest_asyncio.apply()

### Enclosing multi-process code within `if __name__ == "__main__"`

When running multi-process Python code on macOS or Windows, as is the case when the 
concurrent futures worker is selected (i.e. `worker="cf"`), then scripts that execute
the forking code need to be enclosed within an `if __name__ == "__main__"` block, e.g.

In [ ]:
from pydra.tasks.testing import UnsafeDivisionWorkflow
from pydra.engine.submitter import Submitter

# This workflow will fail because we are trying to divide by 0
wf = UnsafeDivisionWorkflow(a=10, b=5, denominator=2)

if __name__ == "__main__":
    with Submitter(worker="cf") as sub:
        result = sub(wf)

This allows the secondary processes to import the script without executing it. Without
such a block Pydra will lock up and not process the workflow. On Linux this is not an
issue due to the way that processes are forked, but is good practice in any case for
code portability.

### Stray leases

While a job runs, the process running it holds a lease on it: a directory named
`<checksum>-<context>.lock` in the cache root. Other processes wanting the same job wait
for the lease to be released, then reuse the job's result. A waiter waits for as long as
the holder keeps renewing its lease, which it does every 30 seconds. If the holder is
killed (for example by `kill -9`, by a debugger, or with its node), the lease stops being
renewed, and another process takes it over once it has not been renewed for `stale_after`
(300 seconds by default), whichever host it runs on. Age is read from the storage's clock,
so hosts whose clocks disagree still agree on it. A holder whose lease was taken over (a
process suspended for longer than `stale_after`) raises `LeaseLost` and does not write its
result over the new holder's.

A wait ends in a `LeaseTimeout` naming the holder's host and pid only when no renewal has
been seen for `timeout` (600 seconds by default) and the lease could not be taken over, for
example when the storage refuses to let this user remove it (`LeaseUnavailable`). To cap
every wait, even behind a live holder, set `max_wait`. All are set on the store:

```python
from pydra.engine.lease import LeaseSettings
from pydra.engine.store import PickleStore

store = PickleStore(cache_root, lease_settings=LeaseSettings(stale_after=60, max_wait=3600, attribute_delay=0))
with Submitter(store=store) as sub:
    ...
```

`attribute_delay=0` suits a cache on local disk; keep the default (60 seconds, NFS's
`acdirmax`) on network storage.

A lock file left by an earlier version of Pydra is taken over by its age, since it is never
renewed. Do not run an earlier Pydra and this one on the same cache at the same time: a job
of the earlier version that runs longer than `stale_after` is run again by this one, which
removes the earlier job's working directory.

## Inspecting errors

### Running in *debug* mode

By default, Pydra will run with the *debug* worker, which executes each task serially
within a single process without use of `async/await` blocks, to allow raised exceptions
to propagate gracefully to the calling code. If you are having trouble with a pipeline,
ensure that `worker=debug` is passed to the submission/execution call (the default).

### Stopping at the first error or raising at the end

The `on_error` argument of the submission/execution call sets what happens when a job
errors. With `on_error="fail_fast"` (the default for synchronous workers such as *debug*)
the first error is raised straight away; jobs already running in a process pool still
finish, but nothing new is scheduled. With `on_error="raise_at_end"` (the default for
other workers) every job that does not depend on an errored job still runs, and the errors
are raised together once the workflow has finished. An errored element of a split node
only stops the downstream elements that read it, so its siblings and their descendants
complete. The error summary of the last run is kept in the Submitter's `errors` list.

### Reading error files

When a task raises an error, it is captured and saved in pickle file named `_error.pklz`
within task's cache directory. For example, when calling the toy `UnsafeDivisionWorkflow`
with a `denominator=0`, the task will fail.

In [ ]:
# This workflow will fail because we are trying to divide by 0
wf = UnsafeDivisionWorkflow(a=10, b=5).split(denominator=[3, 2, 0])

if __name__ == "__main__":
    try:
        with Submitter(worker="cf") as sub:
            result = sub(wf)
    except Exception as e:
        print(e)

The error pickle files can be viewed using the `pydracli crash` command, with the possibility of rerunning and debugging the job. Note that it is
important to use the same Python version to load the files that was used to run the Pydra
workflow

In [ ]:
from pydra.engine.store import PickleStore
from pydra.tasks.testing import Divide
from pydra.utils.general import default_run_cache_root

if __name__ == "__main__":
    divide = Divide(x=15, y=0)
    try:
        # a PickleStore keeps each job's files, _error.pklz among them, in a slot
        divide(cache_root=default_run_cache_root(), store="pickle")
    except Exception:
        pass

    (job_dir,) = PickleStore(default_run_cache_root()).slots(divide._checksum)
    errorfile = job_dir / "_error.pklz"

In [ ]:
%%bash -s "$errorfile"

pydracli crash $1

## Tracing upstream issues

Failures are common in scientific analysis, even for well tested workflows, due to
the novel nature and of scientific experiments and known artefacts that can occur.
Therefore, it is always to sanity-check results produced by workflows. When a problem
occurs in a multi-stage workflow it can be difficult to identify at which stage the
issue occurred.

Currently in Pydra you need to step backwards through the tasks of the workflow, load
the saved task object and inspect its inputs to find the preceding nodes. If any of the
inputs that have been generated by previous nodes are not ok, then you should check the
tasks that generated them in turn. For file-based inputs, you should be able to find
the path of the preceding task's cache directory from the provided file path. However,
for non-file inputs you may need to exhaustively iterate through all the task dirs
in your cache root to find the issue.

For example, in the following example workflow, if a divide by 0 occurs within the division
node of the workflow, then an `float('inf')` will be returned, which will then propagate
through the workflow.

In [ ]:
from pydra.engine.submitter import Submitter
from pydra.tasks.testing import SafeDivisionWorkflow

wf = SafeDivisionWorkflow(a=10, b=5).split(denominator=[3, 2, 0])

if __name__ == "__main__":
    # a PickleStore run cache, whose slots the next cell reads
    with Submitter(worker="cf", store="pickle") as sub:
        result = sub(wf)

print(f"Workflow completed successfully, results saved in: {result.cache_dir}")

To find the task directory where the issue first surfaced, iterate through every task
cache dir and check the results for `float("inf")`s

In [ ]:
import cloudpickle as cp
from pydra.engine.store import PickleStore
from pydra.utils.general import user_cache_root

run_cache = PickleStore(user_cache_root() / "run-cache")

for task_cache_root in run_cache.slots():
    with open(task_cache_root / "_result.pklz", "rb") as f:
        result = cp.load(f)
    if result.outputs is not None:
        for field_name in result.outputs:
            if result.outputs[field_name] == float("nan"):
                print(
                    f"Job {task_cache_root.parent.name!r} produced a NaN value for {field_name!r}"
                )